<a href="https://colab.research.google.com/github/PriyanshuBhunia/classification-ML/blob/main/Adult_MLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ucimlrepo

In [9]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
adult = fetch_ucirepo(id=2)

# data (as pandas dataframes)
X = adult.data.features
y = adult.data.targets

y['income'] = y['income'].apply(lambda x: 1 if '>50K' in x else 0)
print (y)

       income
0           0
1           0
2           0
3           0
4           0
...       ...
48837       0
48838       0
48839       0
48840       0
48841       1

[48842 rows x 1 columns]


/tmp/ipython-input-44176740.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y['income'] = y['income'].apply(lambda x: 1 if '>50K' in x else 0)


In [4]:
import numpy as np

X.replace('?', np.nan, inplace=True)
X.dropna(inplace=True)  # or use imputation if preferred


/tmp/ipython-input-4115453205.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.replace('?', np.nan, inplace=True)
/tmp/ipython-input-4115453205.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.dropna(inplace=True)  # or use imputation if preferred


In [12]:
import numpy as np

X.replace('?', np.nan, inplace=True)
X.dropna(inplace=True)
y = y.loc[X.index]  # sync labels with filtered rows



/tmp/ipython-input-3372765125.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.replace('?', np.nan, inplace=True)
/tmp/ipython-input-3372765125.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.dropna(inplace=True)


In [19]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
X_full = X.copy()

# Identify column types
categorical_cols = X.select_dtypes(include='object').columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# One-hot encode categoricals
X_encoded = pd.get_dummies(X[categorical_cols])

# Scale numericals
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X[numerical_cols]), columns=numerical_cols, index=X.index)

# Combine all features
X_preprocessed = pd.concat([X_scaled, X_encoded], axis=1)

print (X_preprocessed)

            age    fnlwgt  education-num  capital-gain  capital-loss  \
0      0.034201 -1.062295       1.128753      0.142888      -0.21878   
1      0.866417 -1.007438       1.128753     -0.146733      -0.21878   
2     -0.041455  0.245284      -0.438122     -0.146733      -0.21878   
3      1.093385  0.425853      -1.221559     -0.146733      -0.21878   
4     -0.798015  1.407393       1.128753     -0.146733      -0.21878   
...         ...       ...            ...           ...           ...   
48836 -0.419735  0.525154       1.128753     -0.146733      -0.21878   
48837  0.034201  0.243135       1.128753     -0.146733      -0.21878   
48839 -0.041455  1.753613       1.128753     -0.146733      -0.21878   
48840  0.412481 -1.001947       1.128753      0.579985      -0.21878   
48841 -0.268423 -0.071818       1.128753     -0.146733      -0.21878   

       hours-per-week  workclass_Federal-gov  workclass_Local-gov  \
0           -0.078120                  False                False 

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X_preprocessed.values, y.values.ravel(), test_size=0.3, stratify=y, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

In [24]:
import torch
import numpy as np # Import numpy
from torch.utils.data import TensorDataset, DataLoader

def to_tensor_loader(X, y, batch_size=64, shuffle=False):
    # Ensure X is of a numerical type suitable for torch.tensor
    X = X.astype(np.float32) # Explicitly convert to float32
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.long)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=shuffle)

train_loader = to_tensor_loader(X_train, y_train, shuffle=True)
val_loader = to_tensor_loader(X_val, y_val)
test_loader = to_tensor_loader(X_test, y_test)

In [25]:
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=[128, 64], output_dim=2):
        super(MLP, self).__init__()
        layers = []

        dims = [input_dim] + hidden_dims
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.3))  # optional: helps prevent overfitting

        layers.append(nn.Linear(dims[-1], output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


In [26]:
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score

def train(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    return acc, f1


In [27]:
import torch

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize model
input_dim = X_train.shape[1]
model = MLP(input_dim=input_dim).to(device)

# Optimizer and loss
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Training loop
num_epochs = 20
for epoch in range(num_epochs):
    train_loss = train(model, train_loader, optimizer, criterion, device)
    val_acc, val_f1 = evaluate(model, val_loader, device)
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")


Epoch 1/20 | Train Loss: 0.3484 | Val Acc: 0.8502 | Val F1: 0.6739
Epoch 2/20 | Train Loss: 0.3197 | Val Acc: 0.8538 | Val F1: 0.6660
Epoch 3/20 | Train Loss: 0.3156 | Val Acc: 0.8524 | Val F1: 0.6626
Epoch 4/20 | Train Loss: 0.3130 | Val Acc: 0.8507 | Val F1: 0.6464
Epoch 5/20 | Train Loss: 0.3107 | Val Acc: 0.8514 | Val F1: 0.6622
Epoch 6/20 | Train Loss: 0.3066 | Val Acc: 0.8524 | Val F1: 0.6700
Epoch 7/20 | Train Loss: 0.3065 | Val Acc: 0.8512 | Val F1: 0.6721
Epoch 8/20 | Train Loss: 0.3045 | Val Acc: 0.8539 | Val F1: 0.6698
Epoch 9/20 | Train Loss: 0.3024 | Val Acc: 0.8536 | Val F1: 0.6722
Epoch 10/20 | Train Loss: 0.3006 | Val Acc: 0.8514 | Val F1: 0.6567
Epoch 11/20 | Train Loss: 0.2989 | Val Acc: 0.8530 | Val F1: 0.6631
Epoch 12/20 | Train Loss: 0.2984 | Val Acc: 0.8542 | Val F1: 0.6698
Epoch 13/20 | Train Loss: 0.2950 | Val Acc: 0.8523 | Val F1: 0.6719
Epoch 14/20 | Train Loss: 0.2948 | Val Acc: 0.8508 | Val F1: 0.6673
Epoch 15/20 | Train Loss: 0.2930 | Val Acc: 0.8515 | Val 